In [149]:
import pandas as pd
import os
import ast

In [150]:
root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
#root = "Y:/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
outputFolder = root + "DataFrames" + "/"

In [151]:
df = pd.read_csv(root + '/DataFrames/MainDataFrame_Lineages_Augmin.csv')
df.columns

Index(['Unnamed: 0', 'Track_ID', 'Source_ID', 'Splitting_event', 'Lineage',
       'Track_Coordinate_X', 'Track_Coordinate_Y', 'Frame', 'EndPoint_ID',
       'Position', 'Dataset', 'Generation', 'Mother_ID', 'Grandmother_ID',
       'Sister_ID', 'Cell_Cycle_mins', 'Cell_Cycle_hours', 'Seen_sister',
       'Seen_granny'],
      dtype='object')

In [152]:
# Parse mitotic error tables (after FIJI analysis)

error_path_1 = root + "Errors_siHAU6.xlsx"
error_path_2 = root + "Errors_siLUC1.xlsx"

error_paths = [error_path_1, error_path_2]

def parse_error(error_list = error_paths):
    dfs = []
    for path in error_list:
        df = pd.read_excel(path)
        dfs.append(df)
    return pd.concat(dfs)

df_error = parse_error().replace({"T": True, "F": False, "R": False})

# Fix dtypes in evaluation columns 
# (Mitotic death exluded from this list!)

cols = [
    "Laggings", 
    "Massive Missegregation", 
    "DNA bridge", 
    "Misaligned", 
    "Multipolar Divition", 
    'Citokinesis Failure',
    'Slippage',
    'MN' # Check if needed
       ]

# Work on a copy to avoid warnings
bool_df = df_error.loc[:, cols].copy()

# Convert and cast to nullable boolean
bool_df = bool_df.apply(lambda col: col.astype("boolean"))

# Convert values: keep empty as NaN, only "True"/True as True, "False"/False as False
# TODO try to get this to work with the original df 
def to_bool_or_nan(x):
    if x in [True, "True"]:
        return True
    elif x in [False, "False"]:
        return False
    else:
        return np.nan   # preserve empties

bool_df = bool_df.map(to_bool_or_nan)
bool_df.loc[:, "any_true"] = bool_df.any(axis = 1, bool_only = True)

# New column: True if any True, NaN if all are NaN, otherwise False
df_error.loc[:, "any_true"] = bool_df.any(axis = 1)

print(df_error.groupby("Dataset")["any_true"].value_counts(dropna = False))

df_error.head()

Dataset   any_true
20250724  False       215
          True         87
20250807  False       232
          True         61
Name: count, dtype: int64


/var/folders/lr/cb1nd6l97696j8chvkss86580000gn/T/ipykernel_41072/2709921327.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_error = parse_error().replace({"T": True, "F": False, "R": False})


,Source_ID,Dataset,Position,Laggings,Massive Missegregation,DNA bridge,Misaligned,MN,Multipolar Divition,Citokinesis Failure,Slippage,Mitotic Death,Comments,any_true
0,8825,20250724,6,True,False,False,False,True,False,True,False,False,check again,True
1,10714,20250724,6,False,False,False,False,False,False,False,False,False,NaN,False
2,9755,20250724,6,False,False,False,False,False,False,False,False,False,NaN,False
3,9330,20250724,6,False,False,False,False,False,False,False,False,False,NaN,False
4,9630,20250724,6,False,False,False,False,False,False,False,False,False,NaN,False


In [153]:
# parse Death dataframes after daughter analysis (FIJI)

def getdeaths(filepath, position, dataset):
    deaths = pd.read_csv(filepath, usecols = ["Death", "EndPoint_ID", "Track_ID", "Downstream_IDs"])
    deaths["Position"] = position
    deaths["Dataset"] = dataset
    
    return deaths

# Define metadata for each file
deaths_info = [
    {"date": "20250724", "position": 2, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 3, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 4, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 5, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 6, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 7, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 8, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 9, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    
    {"date": "20250807", "position": 2, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 3, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 4, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 5, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 6, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 7, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 8, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},

    {"date": "20250814", "position": 2, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 3, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 4, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 5, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 6, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 7, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 8, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
   # {"date": "20250814", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
]

# Generate and combine dataframes
df_deaths_list = []

for info in deaths_info:
    folder = f"{root}/{info['date']}/Position_{info['position']}_si{info['target']}/Concatenated"
    filepath = os.path.join(folder, info["file"])
    df_single = getdeaths(filepath, info["position"], info["date"])
    df_deaths_list.append(df_single)

df_deaths = pd.concat(df_deaths_list, ignore_index = True)
df_deaths["Dataset"] = df_deaths.Dataset.astype(int)
df_deaths["EndPoint_ID"] = df_deaths.EndPoint_ID.astype(str)
print(df_deaths.shape)
df_deaths = df_deaths[df_deaths["Death"] == True]
print(df_deaths.shape)


df["EndPoint_ID"] = df["EndPoint_ID"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Collect all endpoint IDs from df (flatten the lists)
all_df_endpoints = set(e for lst in df["EndPoint_ID"] if isinstance(lst, list) for e in lst)

# Collect all death endpoint IDs
all_dead_endpoints = set(df_deaths.loc[df_deaths["Death"], "EndPoint_ID"])

print("Number of endpoints in df:", len(all_df_endpoints))
print("Number of death endpoints:", len(all_dead_endpoints))
print("Number of overlaps:", len(all_df_endpoints & all_dead_endpoints))

(414, 6)
(334, 6)
Number of endpoints in df: 380
Number of death endpoints: 331
Number of overlaps: 307


In [154]:
print(df["EndPoint_ID"].head(10))
print(df["EndPoint_ID"].map(type).value_counts())

0        []
1        []
2    [2067]
3    [2143]
4        []
5        []
6    [2846]
7    [3125]
8    [2725]
9        []
Name: EndPoint_ID, dtype: object
EndPoint_ID
<class 'list'>    1203
Name: count, dtype: int64


In [155]:
def count_dead_offspring(row, df_deaths):
    # Filter deaths for same Track_ID & Dataset
    deaths_subset = df_deaths[
        (df_deaths["Track_ID"] == row["Track_ID"]) &
        (df_deaths["Dataset"] == row["Dataset"]) &
        (df_deaths["Death"])
    ]
    dead_endpoints = set(deaths_subset["EndPoint_ID"])

    # Count how many of this row's endpoints are in the dead set
    endpoint_list = row["EndPoint_ID"]
    if isinstance(endpoint_list, list):
        return sum(e in dead_endpoints for e in endpoint_list)
    return 0

df["N_dead_offspring"] = df.apply(count_dead_offspring, axis = 1, df_deaths = df_deaths)

print(df[["N_dead_offspring", "Source_ID"]])

      N_dead_offspring  Source_ID
0                    0       2989
1                    0       3052
2                    0       2005
3                    1       2121
4                    0       2902
...                ...        ...
1198                 2       2425
1199                 0       4820
1200                 0       5139
1201                 0       5634
1202                 0       4781

[1203 rows x 2 columns]


In [156]:
def getonsets(filepath, position, dataset):
    onsets = pd.read_csv(filepath)
    onsets["Source_ID"] = onsets.Label.str.split(":").str.get(1).astype(int)
    onsets["Position"] = position
    onsets["Dataset"] = dataset
    onsets.rename(columns = {"Slice": "Frame_onset"}, inplace = True)
    return onsets

# Define metadata for each file
onsets_info = [
    {"date": "20250724", "position": 2, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 3, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 4, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 5, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 6, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 7, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 8, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 9, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    
    {"date": "20250807", "position": 2, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 3, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 4, "target": "LUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 5, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 6, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 7, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 8, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},

    {"date": "20250814", "position": 2, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 3, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 4, "target": "LUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 5, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 6, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 7, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 8, "target": "HAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
   # {"date": "20250814", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
]

# Generate and combine dataframes
df_onsets_list = []

for info in onsets_info:
    folder = f"{root}/{info['date']}/Position_{info['position']}_si{info['target']}/Concatenated"
    filepath = os.path.join(folder, info["file"])
    df_single = getonsets(filepath, info["position"], info["date"])
    df_onsets_list.append(df_single)

df_onsets = pd.concat(df_onsets_list, ignore_index = True)
df_onsets["Dataset"] = df_onsets.Dataset.astype(int)

In [157]:
df["Frame"] = df.Frame.astype(int) + 1 #otherwise the trackmate and mitotic timing frames are not aligned

df = df.merge(df_onsets, how = "outer") 

In [158]:
def getInterphaseDuration(x, dataframe = df):
    # should be the same as cell_cycle - mitotic duration
    daughter_time = x.Frame_onset
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        if mother_row["Frame"].shape[0] == 1:   
            mother_time = mother_row["Frame"].item()
            interphase_duration = (int(daughter_time) - int(mother_time)) * 7
            return interphase_duration
        else:
            return None


def getMitoticDurationMother(x, dataframe = df):
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        
        if len(mother_row) == 1:
            return mother_row.iloc[0].Mitotic_duration_mins
        else:
            return None

def getMetaphaseArrest(x):
    if x < 30:
        return "000-030 min"
    elif (x < 60) & (x > 30):
        return "030-060"
    elif (x < 90) & (x > 60):
        return "060-090"
    elif (x < 120) & (x > 90):
        return "090-120"
    else:
        return "> 120 min"

def getMetaphaseArrest_mother(x):
    if x > 0:
        if x < 30:
            return "000-030 min"
        elif (x < 60) & (x > 30):
            return "030-060"
        elif (x < 90) & (x > 60):
            return "060-090"
        elif (x < 120) & (x > 90):
            return "090-120"
        else:
            return "> 120 min"
    else:
        pass

df["Frames_in_mitosis"] = (df.Frame - df.Frame_onset.astype(int))
df["Mitotic_duration_mins"] = df.Frames_in_mitosis * 7

df["Interphase_duration_mins"] = df.apply(getInterphaseDuration, axis = 1) 
df["Interphase_duration_hrs"] = df.Interphase_duration_mins / 60
df["Mitotic_duration_hrs"] = df.Mitotic_duration_mins / 60

df["Mother_Mitotic_duration_mins"] = df.apply(getMitoticDurationMother, axis = 1)
df["Mother_arrested"] = df.Mother_Mitotic_duration_mins.apply(getMetaphaseArrest_mother)
df["Metaphase_arrested"] = df.Mitotic_duration_mins.apply(getMetaphaseArrest)
df["Frame_onset_hrs"] = df.Frame_onset * 7 / 60 # Augmin depletion time at NEB
df["Frame_hrs"] = df.Frame * 7 / 60 # Augmin depletion time at anaphase

# Create Bins (one bin for every 12 h of depletion, binning cells with the same onset of mitotis)
interval_range_depletion = pd.interval_range(start = 0, freq = 12, end = 96) 
df['Augmin_depletion_bin'] = pd.cut(df['Frame_onset_hrs'], bins = interval_range_depletion).astype(str)

# Create Bins (one bin for every 30 min of mitotic duration) # kind of is the same as "Metaphase_arrested"
interval_range_mitosis = pd.interval_range(start = 0, freq = 30, end = 900) 
df['Mitotic_duration_bin'] = pd.cut(df['Mitotic_duration_mins'], bins = interval_range_mitosis).astype(str)

df["Unique_Track_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Track_ID.astype(str)
df["Unique_Source_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Source_ID.astype(str)

# Drop annotations with mistakes
df = df.drop(df[df.Cell_Cycle_hours < 1].index)
df = df.drop(df[df.Mitotic_duration_mins < 0].index)
df = df.drop(df[df.Mother_Mitotic_duration_mins < 0].index)

df.head()

,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,EndPoint_ID,Position,...,Mitotic_duration_hrs,Mother_Mitotic_duration_mins,Mother_arrested,Metaphase_arrested,Frame_onset_hrs,Frame_hrs,Augmin_depletion_bin,Mitotic_duration_bin,Unique_Track_ID,Unique_Source_ID
0,0,0,2004,True,2004,811.656027,757.794218,297,"[2029, 2032]",8,...,1.283333,NaN,None,060-090,33.366667,34.650000,"(24, 36]","(60, 90]",2025081480,2025081482004
1,0,0,2005,True,2005,666.091036,713.323699,118,[2067],2,...,1.050000,NaN,None,060-090,12.716667,13.766667,"(12, 24]","(60, 90]",2025072420,2025072422005
2,0,0,2008,True,2008,1066.878136,654.766245,59,[2025],2,...,0.700000,NaN,None,030-060,6.183333,6.883333,"(0, 12]","(30, 60]",2025080720,2025080722008
3,1,0,2026,True,2004.2026,783.482158,819.666245,736,"[2029, 2032]",8,...,7.583333,77.0,060-090,> 120 min,78.283333,85.866667,"(72, 84]","(450, 480]",2025081480,2025081482026
4,1,0,2027,True,2005.2027,662.224034,670.786681,534,[2067],2,...,1.283333,63.0,060-090,060-090,61.016667,62.300000,"(60, 72]","(60, 90]",2025072420,2025072422027


In [159]:
def getCondition(x):
    if x.Dataset == 20250724:
        if x.Position < 6:
            return "siLUC1"
        else:
            return "siHAUS6"

    elif x.Dataset == 20250807 or x.Dataset == 20250814:
        if x.Position < 5:
            return "siLUC1"
        else:
            return "siHAUS6"

df["Condition"] = df.apply(getCondition, axis = 1)

In [160]:
# Final (outer) merge with error dataframe

df = df.merge(df_error, how = "outer")

In [161]:
# Correct daughter deaths with mitotic deaths

def correct_death(x):
    if x["Mitotic Death"] == True:
        return x.N_dead_offspring - 2
    else:
        return x.N_dead_offspring

def death_bool(x):
    if x > 0:
        return True
    else:
        return False

df["N_dead_offspring"] = df.apply(correct_death, axis = 1) 
df["N_dead_offspring_bool"] = df.N_dead_offspring.apply(death_bool)

In [162]:
df.to_csv(outputFolder + "MainDataFrame_MitoticStopwatch_Augmin.csv")

print("Finished.")

Finished.
